# ACE inhibitor classification workflow — cleaned and reproducible

**Purpose:** Train an ACE inhibitor classifier from ChEMBL IC₅₀ records using a pre-specified threshold, compare baseline models with scaffold-grouped cross-validation, evaluate the selected model once on a compound-disjoint external set, and optionally screen compound libraries.

## Design decisions
- Training source: `CHEMBL1808`; external source: `CHEMBL2096989`.
- Default activity definition: **active if IC₅₀ ≤ 25 nM** (set in `ace_config.IC50_CUTOFF_NM`). Change this only before running the complete workflow and record the rationale.
- **Biochemical / binding assays only** (ChEMBL `assay_type == "B"`); cell, tissue and in-vivo records are excluded.
- **Censored records:** `>` / `>=` IC₅₀ values ≥ 10,000 nM are kept as *inactive*. `<` records and smaller `>` bounds are dropped. Censored compounds have no pChEMBL, so the regressor (notebook 2) ignores them.
- **ACE-inhibitor prodrugs** (ester / thioester / phosphinate-ester chemotypes such as enalapril, ramipril, fosinopril, zofenopril) are removed from the training and external sets and from the screening libraries; removed rows are written to CSV for review.
- All of this preparation lives in `ace_config.prepare_chembl_activity`, shared by every notebook.
- Exact canonical-SMILES overlaps are removed from the external evaluation set.
- Morgan fingerprints: radius 2, 2,048 bits.
- Model selection uses training-set cross-validation only. The external set is not used to select models or thresholds.
- Scaffold-aware CV is a more stringent test than random splitting. Acyclic molecules have no Bemis–Murcko scaffold, so each acyclic compound receives its own group rather than putting all acyclic compounds into one artificial shared group. This does not guarantee that acyclic analogues are chemically dissimilar.
- External evaluation is small and imbalanced; report the confusion matrix and numbers of true/false positives and negatives alongside summary metrics.

Run cells from top to bottom in a fresh Colab runtime.

In [2]:
# 1. Environment, Drive, project configuration, and parameters

# Dependencies: run notebook 0 with INSTALL_COLAB_CPU_DEPENDENCIES = True once per fresh Colab runtime.
%pip install rdkit
from pathlib import Path
import os
import sys
import json
import csv
import copy
import warnings
import importlib.util

import numpy as np
import pandas as pd
import sklearn
import rdkit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, f1_score, confusion_matrix, classification_report,
    brier_score_loss, log_loss
)
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
RDLogger.DisableLog("rdApp.*")  # silences per-molecule standardizer log lines (they bloated the notebook to 16 MB)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/ACE_Workflow")
except ImportError:
    ROOT = Path(os.environ.get("ACE_WORKFLOW_ROOT", Path.cwd())).resolve()

if not (ROOT / "ace_config.py").exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / "ace_config.py").exists()), ROOT)

if not (ROOT / "ace_config.py").exists():
    raise FileNotFoundError(
        f"Could not find ace_config.py under {ROOT}. "
        "Mount Drive or set ACE_WORKFLOW_ROOT to the ACE_Workflow folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["ACE_WORKFLOW_ROOT"] = str(ROOT)

import ace_config

# Study settings — pre-specify these before model comparison.
IC50_CUTOFF_NM = float(ace_config.IC50_CUTOFF_NM)  # single source of truth: ace_config.py
N_SPLITS = 5
RANDOM_STATE = int(ace_config.RANDOM_STATE)
FP_RADIUS = 2
FP_SIZE = 2048
PROBABILITY_THRESHOLD = 0.50  # descriptive default; do not tune on external data

TRAIN_PATH = Path(ace_config.INPUTS["CHEMBL1808"])
EXTERNAL_PATH = Path(ace_config.INPUTS["CHEMBL2096989"])

# Use configured project output paths when available; otherwise use stable defaults.
PATHS = getattr(ace_config, "PATHS", {})
OUT = Path(PATHS.get("ML_CLASSIFICATION", ROOT / "results" / "ML_CLASSIFICATION"))
PROCESSED_DIR = Path(PATHS.get("DATA_PROCESSED", ROOT / "data" / "processed"))
MODEL_DIR = Path(PATHS.get("ML_MODELS", ROOT / "models"))
for folder in (OUT, PROCESSED_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

for label, path in [("Training", TRAIN_PATH), ("External", EXTERNAL_PATH)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} input file does not exist: {path}")

print("Project root:", ROOT)
print("Training file:", TRAIN_PATH)
print("External file:", EXTERNAL_PATH)
print("IC50 cutoff (nM):", IC50_CUTOFF_NM)
print("RDKit:", rdkit.__version__)
print("scikit-learn:", sklearn.__version__)
print("Output directory:", OUT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 46.0 MB/s  0:00:00 eta 0:00:01
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ACE_Workflow
Training file: /content/drive/MyDrive/ACE_Workflow/data/raw/CHEMBL1808.csv
External file: /content/drive/MyDrive/ACE_Workflow/data/raw/CHEMBL2096989.csv
IC50 cutoff (nM): 25.0
RDKit: 2026.09.1
scikit-learn: 1.6.1
Output directory: /content/drive/MyDrive/ACE_Workflow/ml_model/results/ic50_classifier


In [3]:
# 2. ChEMBL activity preparation (shared implementation in ace_config.py)
# Rules: IC50 only -> biochemical/binding assays only -> exact "=" values plus ">=10,000 nM" censored inactives
# -> median per standardized compound -> ACE-inhibitor prodrugs removed.

train, train_prep_diag = ace_config.prepare_chembl_activity(
    TRAIN_PATH, IC50_CUTOFF_NM, "Training CHEMBL1808", return_diagnostics=True
)
external_original, external_prep_diag = ace_config.prepare_chembl_activity(
    EXTERNAL_PATH, IC50_CUTOFF_NM, "External CHEMBL2096989", return_diagnostics=True
)

# Keep the removed prodrugs for review.
removed_prodrugs = pd.concat([
    train_prep_diag["prodrugs_removed_table"].assign(dataset="CHEMBL1808"),
    external_prep_diag["prodrugs_removed_table"].assign(dataset="CHEMBL2096989"),
], ignore_index=True)
removed_prodrugs.to_csv(OUT / "prodrugs_removed_from_chembl.csv", index=False)
print(f"\nProdrugs removed: {len(removed_prodrugs)} (see prodrugs_removed_from_chembl.csv)")

print("\nPrepared shapes:", train.shape, external_original.shape)
display(train.head())
display(external_original.head())



Training CHEMBL1808 preparation diagnostics
  raw_rows: 967
  rows_after_ic50_filter: 967
  biochemical_filter: assay_type == B (Binding) [column 'Assay Type']
  assay_breakdown_before_filter: {'B': 831, 'F': 133, 'A': 3}
  rows_after_assay_filter: 831
  exact_rows: 715
  censored_gt_rows_total: 89
  censored_gt_rows_kept_as_inactive: 48
  rows_dropped_other_relations: 27
  valid_measurement_rows: 763
  unique_compounds_before_prodrug_filter: 631
  prodrugs_removed: 41
  unique_compounds: 590
  exact_compounds: 546
  censored_inactive_compounds: 44
  active_compounds: 156
  inactive_compounds: 434
  active prevalence: 26.4%

External CHEMBL2096989 preparation diagnostics
  raw_rows: 55
  rows_after_ic50_filter: 55
  biochemical_filter: assay_type == B (Binding) [column 'Assay Type']
  assay_breakdown_before_filter: {'B': 55}
  rows_after_assay_filter: 55
  exact_rows: 43
  censored_gt_rows_total: 2
  censored_gt_rows_kept_as_inactive: 0
  rows_dropped_other_relations: 10
  valid_measu

,canonical_smiles,ic50_nM,pchembl,n_measurements,ic50_censored,active_ic50,scaffold
0,CC(=O)CN1C(=O)/C(=N\C(=S)NCc2cc3ccccc3cc2O)c2c...,112.201845,6.95,1,False,0,O=C1Nc2ccccc2C1=NC(=S)NCc1ccc2ccccc2c1
1,CC(=O)C[C@@]1(O)C(=O)C2=C3O[C@H](C)C(C)(C)[C@]...,3090.295433,5.51,1,False,0,O=C1CC(=O)c2cccc3c2C1=C1OCCC1C3=O
2,CC(=O)C[C@]1(O)C(=O)C2=C3O[C@H](C)C(C)(C)[C@]3...,11220.184543,4.95,1,False,0,O=C1CC(=O)c2cccc3c2C1=C1OCCC1C3=O
3,CC(=O)N1C(=O)/C(=N\C(=S)NCc2cc3ccccc3cc2O)c2cc...,10.715193,7.97,1,False,1,O=C1Nc2ccccc2C1=NC(=S)NCc1ccc2ccccc2c1
4,CC(=O)N1CCCC1C(NC(C)C(=O)N1CCCC1C(=O)O)C(=O)O,9.332543,8.03,1,False,1,O=C(CNCC1CCCN1)N1CCCC1


,canonical_smiles,ic50_nM,pchembl,n_measurements,ic50_censored,active_ic50,scaffold
0,CC(C)[C@H](S)C(=O)NC1CCc2cccc3c2N(C1=O)[C@H](C...,30.199517,7.52,1,False,0,O=C1CCCc2cccc3c2N1CC3
1,CC(C)[C@H](S)C(=O)N[C@@H](Cc1ccc(-c2ccccc2)cc1...,316.227766,6.50,1,False,0,O=C(CCc1ccc(-c2ccccc2)cc1)N1CCCC1
2,CC(C)[C@H](S)C(=O)N[C@@H](Cc1ccc(-c2ccccc2)cc1...,100.000000,7.00,1,False,0,c1ccc(-c2ccccc2)cc1
3,CC(C)[C@H](S)C(=O)N[C@@H](Cc1ccc(-c2ccccc2)cc1...,61.659500,7.21,1,False,0,O=C(CCc1ccc(-c2ccccc2)cc1)NCCc1ccccc1
4,CC(C)[C@H](S)C(=O)N[C@@H](Cc1ccc(-c2ccccc2)cc1...,173.780083,6.76,1,False,0,c1ccc(-c2ccccc2)cc1


In [4]:
# 3. Dataset integrity, overlap, and label-consistency diagnostics

REQUIRED_COLUMNS = {"canonical_smiles", "ic50_nM", "active_ic50", "scaffold", "ic50_censored"}

def validate_dataset(df, name):
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{name}: missing columns: {sorted(missing)}")
    if df.empty:
        raise ValueError(f"{name}: dataset is empty.")
    if df["canonical_smiles"].isna().any():
        raise ValueError(f"{name}: missing canonical SMILES.")
    if df["canonical_smiles"].duplicated().any():
        raise ValueError(f"{name}: duplicate canonical SMILES.")
    if df["ic50_nM"].isna().any() or (df["ic50_nM"] <= 0).any():
        raise ValueError(f"{name}: invalid IC50 values.")
    if not set(df["active_ic50"].unique()).issubset({0, 1}):
        raise ValueError(f"{name}: activity labels must be 0/1.")
    if df["active_ic50"].nunique() != 2:
        warnings.warn(f"{name}: only one activity class is present.")

    print(f"\n{name}")
    print("  Shape:", df.shape)
    print("  Missing required fields:", sorted(missing))
    print("  Duplicate SMILES:", int(df["canonical_smiles"].duplicated().sum()))
    print("  Class counts:", df["active_ic50"].value_counts().sort_index().to_dict())
    print("  Censored (>= 10,000 nM) inactive compounds:", int(df["ic50_censored"].sum()))
    print("  Unique Murcko grouping labels:", df["scaffold"].nunique())

validate_dataset(train, "Training dataset")
validate_dataset(external_original, "Original external dataset")

train_smiles = set(train["canonical_smiles"])
external_smiles = set(external_original["canonical_smiles"])
overlap_smiles = train_smiles & external_smiles

if overlap_smiles:
    train_overlap = train.loc[
        train["canonical_smiles"].isin(overlap_smiles),
        ["canonical_smiles", "ic50_nM", "active_ic50"]
    ].rename(columns={"ic50_nM": "train_ic50_nM", "active_ic50": "train_active"})

    ext_overlap = external_original.loc[
        external_original["canonical_smiles"].isin(overlap_smiles),
        ["canonical_smiles", "ic50_nM", "active_ic50"]
    ].rename(columns={"ic50_nM": "external_ic50_nM", "active_ic50": "external_active"})

    overlap_details = train_overlap.merge(
        ext_overlap, on="canonical_smiles", validate="one_to_one"
    )
    overlap_details["label_conflict"] = (
        overlap_details["train_active"] != overlap_details["external_active"]
    )
else:
    overlap_details = pd.DataFrame(columns=[
        "canonical_smiles", "train_ic50_nM", "train_active",
        "external_ic50_nM", "external_active", "label_conflict"
    ])

print("\nExact canonical-SMILES overlap:", len(overlap_smiles))
if not overlap_details.empty:
    print("Label conflicts among overlapping compounds:",
          int(overlap_details["label_conflict"].sum()))
    display(overlap_details)

external_independent = external_original.loc[
    ~external_original["canonical_smiles"].isin(train_smiles)
].copy().reset_index(drop=True)

assert not (
    set(train["canonical_smiles"]) & set(external_independent["canonical_smiles"])
), "Exact compound overlap remains in external evaluation set."

print("\nCompound-disjoint external evaluation set")
print("  Compounds:", len(external_independent))
print("  Class counts:",
      external_independent["active_ic50"].value_counts().sort_index().to_dict())
print("  Active prevalence: {:.1%}".format(
    external_independent["active_ic50"].mean()
))
print("  Training scaffolds:", train["scaffold"].nunique())
print("  External scaffolds:", external_independent["scaffold"].nunique())
print("  Scaffolds shared across datasets:",
      len(set(train["scaffold"]) & set(external_independent["scaffold"])))

# Save the exact prepared datasets and diagnostics for traceability.
train.to_csv(PROCESSED_DIR / "chembl1808_ic50_clean.csv", index=False)
external_original.to_csv(
    PROCESSED_DIR / "chembl2096989_ic50_original_clean.csv", index=False
)
external_independent.to_csv(
    PROCESSED_DIR / "chembl2096989_ic50_external_disjoint.csv", index=False
)
overlap_details.to_csv(OUT / "external_overlap_diagnostics.csv", index=False)

print("\nPrepared datasets saved to:", PROCESSED_DIR)


Training dataset
  Shape: (590, 7)
  Missing required fields: []
  Duplicate SMILES: 0
  Class counts: {0: 434, 1: 156}
  Censored (>= 10,000 nM) inactive compounds: 44
  Unique Murcko grouping labels: 283

Original external dataset
  Shape: (41, 7)
  Missing required fields: []
  Duplicate SMILES: 0
  Class counts: {0: 33, 1: 8}
  Censored (>= 10,000 nM) inactive compounds: 0
  Unique Murcko grouping labels: 21

Exact canonical-SMILES overlap: 5
Label conflicts among overlapping compounds: 1


,canonical_smiles,train_ic50_nM,train_active,external_ic50_nM,external_active,label_conflict
0,C[C@@H](CC(=O)[C@@H](Cc1ccccc1)NC(=O)c1ccccc1)...,3162.277660,0,3162.277660,0,False
1,C[C@H](CC(=O)[C@H](Cc1ccccc1)NC(=O)c1ccccc1)C(...,1.000000,1,1.000000,1,False
2,C[C@H](CS)C(=O)N1CCC[C@H]1C(=O)O,20.422792,1,8.912509,1,False
3,O=C(N[C@@H](Cc1ccccc1)C(=O)CCC(=O)N1CCC[C@H]1C...,40.602871,0,3.235937,1,True
4,O=C(N[C@@H](Cc1ccccc1)C(=O)NCC(=O)N1CCC[C@H]1C...,9332.543008,0,9332.543008,0,False



Compound-disjoint external evaluation set
  Compounds: 36
  Class counts: {0: 31, 1: 5}
  Active prevalence: 13.9%
  Training scaffolds: 283
  External scaffolds: 19
  Scaffolds shared across datasets: 5

Prepared datasets saved to: /content/drive/MyDrive/ACE_Workflow/data/processed


In [5]:
# 4. Morgan fingerprint generation and scaffold-group diagnostics

fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE
)

def smiles_to_fp_array(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES encountered during fingerprinting: {smiles}")
    fp = fp_generator.GetFingerprint(mol)
    arr = np.zeros(FP_SIZE, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def fingerprint_matrix(smiles_series):
    if len(smiles_series) == 0:
        raise ValueError("Cannot generate fingerprints for an empty dataset.")
    return np.vstack([smiles_to_fp_array(s) for s in smiles_series])

X_train = fingerprint_matrix(train["canonical_smiles"])
y_train = train["active_ic50"].astype(int).to_numpy()
groups = train["scaffold"].astype(str).to_numpy()

X_external = fingerprint_matrix(external_independent["canonical_smiles"])
y_external = external_independent["active_ic50"].astype(int).to_numpy()

assert X_train.shape == (len(train), FP_SIZE)
assert X_external.shape == (len(external_independent), FP_SIZE)
assert np.isfinite(X_train).all() and np.isfinite(X_external).all()
assert set(np.unique(y_train)).issubset({0, 1})
assert len(np.unique(y_train)) == 2
assert len(np.unique(groups)) >= N_SPLITS

print("Training fingerprint matrix:", X_train.shape, X_train.dtype)
print("External fingerprint matrix:", X_external.shape, X_external.dtype)
print("Training class counts (0=inactive, 1=active):",
      np.bincount(y_train, minlength=2))
print("Unique training CV groups:", len(np.unique(groups)))

scaffold_summary = (
    train.groupby(["scaffold", "active_ic50"])
    .size().unstack(fill_value=0)
    .rename(columns={0: "inactive", 1: "active"})
)
for col in ["inactive", "active"]:
    if col not in scaffold_summary:
        scaffold_summary[col] = 0
scaffold_summary["total"] = scaffold_summary["inactive"] + scaffold_summary["active"]
print("\nLargest scaffold groups:")
display(scaffold_summary.sort_values("total", ascending=False).head(15))

Training fingerprint matrix: (590, 2048) uint8
External fingerprint matrix: (36, 2048) uint8
Training class counts (0=inactive, 1=active): [434 156]
Unique training CV groups: 283

Largest scaffold groups:


active_ic50,inactive,active,total
scaffold,,,
c1ccccc1,67,2,69
C1CCNC1,25,3,28
O=C(NC(Cc1ccccc1)C(=O)CCC(=O)N1CCCC1)c1ccccc1,9,4,13
c1ccc(-c2ccccc2)cc1,10,2,12
O=C(CCc1ccccc1)NCCc1ccccc1,10,0,10
c1ccc2[nH]ccc2c1,10,0,10
O=C1CCCc2ccccc2N1,1,8,9
C(=Cc1ccccc1)c1ccccc1,7,0,7
O=C(CCc1ccccc1)NC1CCc2ccccc2NC1=O,5,2,7


In [6]:
# 5. Define baseline classifiers

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": None,
    "SVM (RBF)": make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf", C=1.0, class_weight="balanced",
            probability=True, random_state=RANDOM_STATE
        )
    ),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=3000, class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.5,
        random_state=RANDOM_STATE
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=7, weights="distance")
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gaussian NB": GaussianNB(),
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128,), max_iter=500,
            early_stopping=True, random_state=RANDOM_STATE
        )
    )
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
except ImportError:
    models.pop("XGBoost")
    print("XGBoost is not installed; proceeding without it.")

assert models, "No models are configured."
print("Models to evaluate:", list(models))

Models to evaluate: ['Random Forest', 'XGBoost', 'SVM (RBF)', 'Logistic Regression', 'AdaBoost', 'KNN', 'Decision Tree', 'Gaussian NB', 'MLP']


In [7]:
# 6. Scaffold-grouped cross-validation — training data only

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)
splits = list(cv.split(X_train, y_train, groups=groups))

# Explicit diagnostics: group leakage, fold sizes, and class composition.
fold_diagnostics = []
for fold_no, (tr_idx, va_idx) in enumerate(splits, start=1):
    shared_groups = set(groups[tr_idx]) & set(groups[va_idx])
    if shared_groups:
        raise AssertionError(f"Scaffold leakage in fold {fold_no}.")

    train_classes = np.bincount(y_train[tr_idx], minlength=2)
    valid_classes = np.bincount(y_train[va_idx], minlength=2)

    fold_diagnostics.append({
        "fold": fold_no,
        "train_n": len(tr_idx),
        "validation_n": len(va_idx),
        "train_inactive": int(train_classes[0]),
        "train_active": int(train_classes[1]),
        "validation_inactive": int(valid_classes[0]),
        "validation_active": int(valid_classes[1]),
        "validation_groups": len(set(groups[va_idx])),
        "scaffold_leakage_groups": len(shared_groups),
        "validation_has_both_classes": len(np.unique(y_train[va_idx])) == 2
    })

fold_diagnostics_df = pd.DataFrame(fold_diagnostics)
display(fold_diagnostics_df)

if not fold_diagnostics_df["validation_has_both_classes"].all():
    warnings.warn(
        "At least one validation fold contains a single class. "
        "ROC-AUC is undefined for that fold; interpret model comparison cautiously."
    )

# Cross-validation predictions are generated independently in each fold.
# External data are not used here.
from sklearn.metrics import precision_score, recall_score

cv_rows = []
oof_results = {}

for model_name, base_model in models.items():
    oof_probability = np.full(len(y_train), np.nan, dtype=float)
    oof_prediction = np.full(len(y_train), -1, dtype=int)
    fold_rows = []

    for fold_no, (tr_idx, va_idx) in enumerate(splits, start=1):
        estimator = clone(base_model)
        estimator.fit(X_train[tr_idx], y_train[tr_idx])

        prob = estimator.predict_proba(X_train[va_idx])[:, 1]
        pred = (prob >= PROBABILITY_THRESHOLD).astype(int)

        oof_probability[va_idx] = prob
        oof_prediction[va_idx] = pred

        tn, fp, fn, tp = confusion_matrix(
            y_train[va_idx], pred, labels=[0, 1]
        ).ravel()

        y_fold = y_train[va_idx]
        fold_rows.append({
            "Model": model_name,
            "Fold": fold_no,
            "N": len(va_idx),
            "ROC_AUC": (
                roc_auc_score(y_fold, prob)
                if len(np.unique(y_fold)) == 2 else np.nan
            ),
            "PR_AUC_AP": average_precision_score(y_fold, prob),
            "Balanced_Accuracy": balanced_accuracy_score(y_fold, pred),
            "MCC": matthews_corrcoef(y_fold, pred),
            "F1": f1_score(y_fold, pred, zero_division=0),
            "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "Specificity": tn / (tn + fp) if tn + fp else np.nan
        })

    if np.isnan(oof_probability).any() or (oof_prediction < 0).any():
        raise AssertionError(f"Incomplete out-of-fold predictions for {model_name}.")

    fold_df = pd.DataFrame(fold_rows)
    tn, fp, fn, tp = confusion_matrix(
        y_train, oof_prediction, labels=[0, 1]
    ).ravel()

    cv_rows.append({
        "Model": model_name,
        "ROC_AUC_pooled_OOF": (
            roc_auc_score(y_train, oof_probability)
            if len(np.unique(y_train)) == 2 else np.nan
        ),
        "PR_AUC_pooled_OOF_AP": average_precision_score(
            y_train, oof_probability
        ),
        "ROC_AUC_fold_mean": fold_df["ROC_AUC"].mean(),
        "ROC_AUC_fold_SD": fold_df["ROC_AUC"].std(),
        "PR_AUC_fold_mean": fold_df["PR_AUC_AP"].mean(),
        "PR_AUC_fold_SD": fold_df["PR_AUC_AP"].std(),
        "Balanced_Accuracy_OOF": balanced_accuracy_score(y_train, oof_prediction),
        "MCC_OOF": matthews_corrcoef(y_train, oof_prediction),
        "F1_OOF": f1_score(y_train, oof_prediction, zero_division=0),
        "Sensitivity_OOF": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity_OOF": tn / (tn + fp) if tn + fp else np.nan,
        "TP_OOF": int(tp), "FP_OOF": int(fp),
        "TN_OOF": int(tn), "FN_OOF": int(fn)
    })

    oof_results[model_name] = {
        "probability": oof_probability,
        "prediction": oof_prediction,
        "fold_metrics": fold_df
    }

cv_results = pd.DataFrame(cv_rows)

# Standardize the PR-AUC column name used downstream
if (
    "PR_AUC_pooled_OOF" not in cv_results.columns
    and "PR_AUC_pooled_OOF_AP" in cv_results.columns
):
    cv_results = cv_results.rename(
        columns={"PR_AUC_pooled_OOF_AP": "PR_AUC_pooled_OOF"}
    )

required_metrics = [
    "PR_AUC_pooled_OOF",
    "ROC_AUC_pooled_OOF",
]

missing_metrics = [
    col for col in required_metrics
    if col not in cv_results.columns
]

if missing_metrics:
    raise KeyError(
        f"Missing expected metrics: {missing_metrics}\n"
        f"Available columns: {cv_results.columns.tolist()}"
    )

cv_results = cv_results.sort_values(
    ["PR_AUC_pooled_OOF", "ROC_AUC_pooled_OOF"],
    ascending=False
).reset_index(drop=True)

cv_results = cv_results.reset_index(drop=True)
display(cv_results)

print("Cross-validation model comparison (training data only):")
display(cv_results.round(4))

,fold,train_n,validation_n,train_inactive,train_active,validation_inactive,validation_active,validation_groups,scaffold_leakage_groups,validation_has_both_classes
0,1,498,92,374,124,60,32,59,0,True
1,2,454,136,333,121,101,35,65,0,True
2,3,526,64,387,139,47,17,35,0,True
3,4,484,106,363,121,71,35,58,0,True
4,5,398,192,279,119,155,37,66,0,True


,Model,ROC_AUC_pooled_OOF,PR_AUC_pooled_OOF,ROC_AUC_fold_mean,ROC_AUC_fold_SD,PR_AUC_fold_mean,PR_AUC_fold_SD,Balanced_Accuracy_OOF,MCC_OOF,F1_OOF,Sensitivity_OOF,Specificity_OOF,TP_OOF,FP_OOF,TN_OOF,FN_OOF
0,Random Forest,0.890649,0.766042,0.890370,0.019524,0.781563,0.027879,0.787753,0.602641,0.698630,0.653846,0.921659,102,34,400,54
1,XGBoost,0.879350,0.703717,0.876977,0.024359,0.729480,0.049122,0.806082,0.636180,0.725424,0.685897,0.926267,107,32,402,49
2,SVM (RBF),0.869705,0.668600,0.867867,0.039119,0.714892,0.068042,0.710431,0.492755,0.583658,0.480769,0.940092,75,26,408,81
3,AdaBoost,0.872681,0.665176,0.866544,0.030160,0.691333,0.036474,0.782997,0.610853,0.697509,0.628205,0.937788,98,27,407,58
4,MLP,0.858192,0.652620,0.848556,0.037888,0.655100,0.059532,0.796009,0.568831,0.688623,0.737179,0.854839,115,63,371,41
5,Logistic Regression,0.837107,0.642084,0.834301,0.023336,0.689989,0.078373,0.780737,0.555898,0.675079,0.685897,0.875576,107,54,380,49
6,KNN,0.791689,0.616277,0.828713,0.032051,0.688633,0.078357,0.690181,0.355245,0.542373,0.615385,0.764977,96,102,332,60
7,Decision Tree,0.760893,0.504984,0.750307,0.033483,0.503267,0.073099,0.758197,0.500263,0.638298,0.673077,0.843318,105,68,366,51
8,Gaussian NB,0.666150,0.392445,0.652984,0.033230,0.400719,0.046472,0.666150,0.338826,0.508251,0.493590,0.838710,77,70,364,79


Cross-validation model comparison (training data only):


,Model,ROC_AUC_pooled_OOF,PR_AUC_pooled_OOF,ROC_AUC_fold_mean,ROC_AUC_fold_SD,PR_AUC_fold_mean,PR_AUC_fold_SD,Balanced_Accuracy_OOF,MCC_OOF,F1_OOF,Sensitivity_OOF,Specificity_OOF,TP_OOF,FP_OOF,TN_OOF,FN_OOF
0,Random Forest,0.8906,0.7660,0.8904,0.0195,0.7816,0.0279,0.7878,0.6026,0.6986,0.6538,0.9217,102,34,400,54
1,XGBoost,0.8793,0.7037,0.8770,0.0244,0.7295,0.0491,0.8061,0.6362,0.7254,0.6859,0.9263,107,32,402,49
2,SVM (RBF),0.8697,0.6686,0.8679,0.0391,0.7149,0.0680,0.7104,0.4928,0.5837,0.4808,0.9401,75,26,408,81
3,AdaBoost,0.8727,0.6652,0.8665,0.0302,0.6913,0.0365,0.7830,0.6109,0.6975,0.6282,0.9378,98,27,407,58
4,MLP,0.8582,0.6526,0.8486,0.0379,0.6551,0.0595,0.7960,0.5688,0.6886,0.7372,0.8548,115,63,371,41
5,Logistic Regression,0.8371,0.6421,0.8343,0.0233,0.6900,0.0784,0.7807,0.5559,0.6751,0.6859,0.8756,107,54,380,49
6,KNN,0.7917,0.6163,0.8287,0.0321,0.6886,0.0784,0.6902,0.3552,0.5424,0.6154,0.7650,96,102,332,60
7,Decision Tree,0.7609,0.5050,0.7503,0.0335,0.5033,0.0731,0.7582,0.5003,0.6383,0.6731,0.8433,105,68,366,51
8,Gaussian NB,0.6661,0.3924,0.6530,0.0332,0.4007,0.0465,0.6661,0.3388,0.5083,0.4936,0.8387,77,70,364,79


In [8]:
# 7. Save CV diagnostics and select the candidate using training data only

cv_results.to_csv(OUT / "scaffold_cv_model_comparison.csv", index=False)
fold_diagnostics_df.to_csv(OUT / "scaffold_cv_fold_diagnostics.csv", index=False)

oof_frames = []
fold_metric_frames = []
for model_name, result in oof_results.items():
    frame = train[[
        "canonical_smiles", "ic50_nM", "active_ic50", "scaffold"
    ]].copy()
    frame["model"] = model_name
    frame["oof_probability_active"] = result["probability"]
    frame["oof_prediction_at_0.50"] = result["prediction"]
    oof_frames.append(frame)
    fold_metric_frames.append(result["fold_metrics"])

oof_predictions_df = pd.concat(oof_frames, ignore_index=True)
fold_metrics_df = pd.concat(fold_metric_frames, ignore_index=True)
oof_predictions_df.to_csv(OUT / "scaffold_cv_oof_predictions.csv", index=False)
fold_metrics_df.to_csv(OUT / "scaffold_cv_fold_metrics.csv", index=False)

# Pre-specified ranking: pooled out-of-fold average precision (PR-AUC/AP),
# with pooled out-of-fold ROC-AUC as a secondary tie-breaker.
best_model_name = cv_results.iloc[0]["Model"]
print("Candidate selected from training-only CV:", best_model_name)
print("Selection metrics:")
display(cv_results.loc[cv_results["Model"] == best_model_name].round(4))

print("\nSaved CV outputs to:", OUT)

Candidate selected from training-only CV: Random Forest
Selection metrics:


,Model,ROC_AUC_pooled_OOF,PR_AUC_pooled_OOF,ROC_AUC_fold_mean,ROC_AUC_fold_SD,PR_AUC_fold_mean,PR_AUC_fold_SD,Balanced_Accuracy_OOF,MCC_OOF,F1_OOF,Sensitivity_OOF,Specificity_OOF,TP_OOF,FP_OOF,TN_OOF,FN_OOF
0,Random Forest,0.8906,0.766,0.8904,0.0195,0.7816,0.0279,0.7878,0.6026,0.6986,0.6538,0.9217,102,34,400,54



Saved CV outputs to: /content/drive/MyDrive/ACE_Workflow/ml_model/results/ic50_classifier


In [9]:
# 8. Refit selected model and perform the external evaluation once

# This is the first and only use of external labels for model performance.
# Do not tune the model or threshold based on these results.
final_model = clone(models[best_model_name])
final_model.fit(X_train, y_train)

external_probability = final_model.predict_proba(X_external)[:, 1]
external_prediction = (
    external_probability >= PROBABILITY_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_external, external_prediction, labels=[0, 1]
).ravel()

external_metrics = {
    "model": best_model_name,
    "N": len(y_external),
    "actual_active": int(y_external.sum()),
    "actual_inactive": int((y_external == 0).sum()),
    "predicted_active": int(external_prediction.sum()),
    "threshold": PROBABILITY_THRESHOLD,
    "ROC_AUC": (
        roc_auc_score(y_external, external_probability)
        if len(np.unique(y_external)) == 2 else np.nan
    ),
    "PR_AUC_average_precision": average_precision_score(
        y_external, external_probability
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_external, external_prediction
    ),
    "MCC": matthews_corrcoef(y_external, external_prediction),
    "F1": f1_score(y_external, external_prediction, zero_division=0),
    "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
    "specificity": tn / (tn + fp) if tn + fp else np.nan,
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    "Brier_score": brier_score_loss(y_external, external_probability),
    "Log_loss": log_loss(
        y_external,
        np.column_stack([1 - external_probability, external_probability]),
        labels=[0, 1]
    )
}

external_metrics_df = pd.DataFrame([external_metrics])
print("External evaluation metrics:")
display(external_metrics_df.round(4))

print("\nExternal confusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_external, external_prediction, labels=[0, 1]))

print("\nExternal classification report:")
print(classification_report(
    y_external, external_prediction,
    labels=[0, 1],
    target_names=["Inactive", "Active"],
    zero_division=0
))

external_predictions_df = external_independent[[
    "canonical_smiles", "ic50_nM", "active_ic50", "scaffold"
]].copy()
external_predictions_df["predicted_probability_active"] = external_probability
external_predictions_df["predicted_active_at_threshold"] = external_prediction

external_metrics_df.to_csv(OUT / "external_evaluation_metrics.csv", index=False)
external_predictions_df.to_csv(
    OUT / "external_compound_predictions.csv", index=False
)

# Save the final model and enough metadata to reproduce its basic setup.
model_path = MODEL_DIR / "ace_ic50_classifier.joblib"
joblib.dump(final_model, model_path)

metadata = {
    "training_database": "CHEMBL1808",
    "external_database": "CHEMBL2096989",
    "activity_definition": f"active if IC50 <= {IC50_CUTOFF_NM} nM; IC50 > / >= {ace_config.CENSORED_INACTIVE_MIN_NM:g} nM kept as inactive",
    "assay_filter": train_prep_diag.get("biochemical_filter", "none"),
    "prodrugs_excluded": ace_config.EXCLUDE_PRODRUGS,
    "training_censored_inactive_compounds": int(train["ic50_censored"].sum()),
    "ic50_cutoff_nM": IC50_CUTOFF_NM,
    "selected_model": best_model_name,
    "selection_method": "training-only scaffold-grouped CV; pooled OOF AP, ROC-AUC tie-breaker",
    "cv_splitter": "StratifiedGroupKFold",
    "n_splits": N_SPLITS,
    "random_state": RANDOM_STATE,
    "fingerprint": {"type": "Morgan", "radius": FP_RADIUS, "n_bits": FP_SIZE},
    "probability_threshold": PROBABILITY_THRESHOLD,
    "training_compounds": len(train),
    "external_compounds_after_exact_overlap_removal": len(external_independent),
    "external_active_count": int(y_external.sum()),
    "external_shared_scaffold_count": len(
        set(train["scaffold"]) & set(external_independent["scaffold"])
    )
}
(OUT / "classifier_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("\nFinal model saved:", model_path)
print("External outputs saved to:", OUT)
print(
    "\nInterpret external metrics cautiously: the external set is small, "
    "has few active compounds, and is not fully scaffold-disjoint."
)

External evaluation metrics:


,model,N,actual_active,actual_inactive,predicted_active,threshold,ROC_AUC,PR_AUC_average_precision,balanced_accuracy,MCC,F1,sensitivity,specificity,TN,FP,FN,TP,Brier_score,Log_loss
0,Random Forest,36,5,31,10,0.5,0.6355,0.1967,0.4548,-0.0697,0.1333,0.2,0.7097,22,9,4,1,0.2073,0.5925



External confusion matrix [rows=true, columns=predicted]:
[[22  9]
 [ 4  1]]

External classification report:
              precision    recall  f1-score   support

    Inactive       0.85      0.71      0.77        31
      Active       0.10      0.20      0.13         5

    accuracy                           0.64        36
   macro avg       0.47      0.45      0.45        36
weighted avg       0.74      0.64      0.68        36


Final model saved: /content/drive/MyDrive/ACE_Workflow/ml_model/saved_models/ace_ic50_classifier.joblib
External outputs saved to: /content/drive/MyDrive/ACE_Workflow/ml_model/results/ic50_classifier

Interpret external metrics cautiously: the external set is small, has few active compounds, and is not fully scaffold-disjoint.


## Optional: screen compound libraries

Run the next cell only after the model has been finalized. Screening probabilities are **model scores, not evidence of experimentally confirmed ACE inhibition**. Tanimoto similarity to training compounds is an applicability-domain signal and should be considered alongside probability.

In [10]:
# 9. Optional compound-library screening
# Missing libraries are skipped with a clear message.
# This cell expects the configured input keys and ace_config.prepare_screening_library.

SOURCE_KEYS = ["FDA_DRUGBANK", "EXPERIMENTAL_ZIAI", "NPASS", "PLANTS_CMAUP"]
screened_tables = []

if not hasattr(ace_config, "prepare_screening_library"):
    print("Screening skipped: ace_config.prepare_screening_library is unavailable.")
else:
    for key in SOURCE_KEYS:
        path = ace_config.INPUTS.get(key)
        if path is None or not Path(path).exists():
            print(f"Skipping missing library {key}: {path}")
            continue

        library = ace_config.prepare_screening_library(Path(path), key)
        if library is None or library.empty:
            print(f"Skipping empty library: {key}")
            continue
        if "canonical_smiles" not in library.columns:
            raise KeyError(f"{key}: screening library lacks canonical_smiles.")

        # Remove invalid standardized structures before fingerprinting.
        library = library.loc[library["canonical_smiles"].notna()].copy()
        if "source" not in library.columns:
            library["source"] = key  # notebook 3 merges on (source, canonical_smiles)
        if ace_config.EXCLUDE_PRODRUGS:
            library, prodrugs = ace_config.split_ace_prodrugs(library)
            prodrugs.to_csv(OUT / f"{key.lower()}_prodrugs_excluded.csv", index=False)
            print(f"{key}: excluded {len(prodrugs)} ACE-inhibitor prodrugs")
        X_library = fingerprint_matrix(library["canonical_smiles"])

        library["ace_active_probability"] = final_model.predict_proba(X_library)[:, 1]
        library["model"] = best_model_name

        # Optional similarity diagnostic, if the project helper is available.
        if hasattr(ace_config, "max_tanimoto_similarity"):
            try:
                library["max_train_tanimoto"] = ace_config.max_tanimoto_similarity(
                    X_library, X_train
                )
            except Exception as exc:
                warnings.warn(
                    f"Could not calculate training similarity for {key}: {exc}"
                )

        library.to_csv(OUT / f"{key.lower()}_ic50_screen.csv", index=False)
        screened_tables.append(library)
        print(f"{key}: screened {len(library)} compounds")

    if screened_tables:
        all_screened = pd.concat(screened_tables, ignore_index=True)
        all_screened.to_csv(OUT / "all_libraries_ic50_screen.csv", index=False)
        display(
            all_screened.sort_values(
                "ace_active_probability", ascending=False
            ).head(25)
        )
        print("Screening outputs saved to:", OUT)
    else:
        print("No available libraries were screened.")

FDA_DRUGBANK: excluded 13 ACE-inhibitor prodrugs
FDA_DRUGBANK: screened 2438 compounds
EXPERIMENTAL_ZIAI: excluded 0 ACE-inhibitor prodrugs
EXPERIMENTAL_ZIAI: screened 7445 compounds
NPASS: excluded 6 ACE-inhibitor prodrugs
NPASS: screened 91920 compounds
PLANTS_CMAUP: excluded 1 ACE-inhibitor prodrugs
PLANTS_CMAUP: screened 57855 compounds


,source,source_id,input_smiles,canonical_smiles,ace_active_probability,model,max_train_tanimoto
1788,FDA_DRUGBANK,Enalaprilat,CC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,CC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,0.943056,Random Forest,1.000000
565,FDA_DRUGBANK,Lisinopril,NCCCCC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,NCCCCC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,0.939695,Random Forest,1.000000
1002,FDA_DRUGBANK,Captopril,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,0.867471,Random Forest,1.000000
63800,NPASS,NPC483990,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,0.867471,Random Forest,1.000000
144495,PLANTS_CMAUP,FAKRSMQSSFJEIM-RQJHMYQMSA-N,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,0.867471,Random Forest,1.000000
22226,NPASS,NPC313640,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,0.695695,Random Forest,1.000000
114585,PLANTS_CMAUP,IFGCUJZIWBUILZ-UHFFFAOYSA-N,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,0.695695,Random Forest,1.000000
13547,NPASS,NPC249334,CCC(C)C(=O)NC1CCC(=O)N(CCc2ccccc2)C1=O,CCC(C)C(=O)NC1CCC(=O)N(CCc2ccccc2)C1=O,0.578324,Random Forest,0.508197
525,FDA_DRUGBANK,Losartan,CCCCc1nc(Cl)c(CO)n1Cc1ccc(-c2ccccc2-c2nn[nH]n2...,CCCCc1nc(Cl)c(CO)n1Cc1ccc(-c2ccccc2-c2nn[nH]n2...,0.502000,Random Forest,0.709677
25772,NPASS,NPC27833,CC(=O)NC1CCC(=O)N(CC(=O)c2ccccc2)C1=O,CC(=O)NC1CCC(=O)N(CC(=O)c2ccccc2)C1=O,0.480105,Random Forest,0.482759


Screening outputs saved to: /content/drive/MyDrive/ACE_Workflow/ml_model/results/ic50_classifier


## Interpretation and reporting checklist

Before writing manuscript claims, verify and report:

1. The IC₅₀ cutoff and unit conversion rules were fixed before model fitting.
2. The number of raw records, retained measurements, unique compounds, and class counts are documented.
3. Duplicate standardized structures and exact train/external overlaps were checked.
4. All scaffold-grouped folds have zero shared group labels; report fold-level class counts and any fold lacking one class.
5. Model selection used training-only cross-validation. External results did not influence model choice or the probability threshold.
6. Report pooled out-of-fold and fold-summary metrics distinctly; they are not interchangeable.
7. Report external TP, FP, TN, FN and class counts. With only a few active external compounds, sensitivity and PR-AUC will be uncertain.
8. Since five scaffolds were previously found to overlap between the training and external sets, describe the external test as **compound-disjoint**, not completely scaffold-independent.
9. Predicted probabilities are not automatically calibrated probabilities, and predictions are not experimental confirmation.